# Colab Setup

Mount Google Drive and pull the latest repo changes before running the experiment cells.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

In [ ]:
import subprocess
from pathlib import Path

REPO_PATH = Path('/content/drive/MyDrive/repos/evolutionary-ai-battle')

if REPO_PATH.exists():
    result = subprocess.run(['git', 'pull'], cwd=REPO_PATH, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
else:
    print(f'Repo path not found: {REPO_PATH}')

# CPC TorchRL Scaffold

This notebook is the initiating experiment entry point. Core logic lives in Python modules; the notebook demonstrates the toy CPC loop, the TorchRL wrapper, and PPO smoke training.

In [ ]:
from pathlib import Path
import json
import sys

EXPERIMENT_ROOT = Path.cwd()
if EXPERIMENT_ROOT.name != "experiment":
    EXPERIMENT_ROOT = Path.cwd() / "experiment"

if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))

from cpc_actions import action_space, decode_action, describe_action
from cpc_env import CPCEnv

print("experiment root:", EXPERIMENT_ROOT)
print("action space:", action_space())

## PR1: Toy CPC Loop

In [ ]:
env = CPCEnv(seed=7, max_steps=8)
obs = env.reset()
move_and_fire = {"move": 6, "aim": 0, "fire": 1}

print(json.dumps(describe_action(move_and_fire), indent=2))
obs, reward, done, info = env.step(move_and_fire)
print("reward", reward, "done", done)
print("decoded", info["decoded_action"])

for _ in range(3):
    action = env.sample_action()
    obs, reward, done, info = env.step(action)
    print({"raw": action, "reward": round(reward, 3), "done": done})
    if done:
        break

print("metrics", json.dumps(env.metrics.summary(), indent=2))
print("trajectory step", json.dumps(env.export_trajectory()["steps"][0], indent=2))

## PR2: TorchRL EnvBase / TensorDict Wrapper

In [ ]:
try:
    import torch
    from torchrl_env import TorchRLCPCEnv
    from torchrl_specs import import_check_env_specs

    torchrl_env = TorchRLCPCEnv(seed=11, max_steps=8)
    print("observation_spec")
    print(torchrl_env.observation_spec)
    print("action_spec")
    print(torchrl_env.action_spec)

    td = torchrl_env.reset()
    td.update(torchrl_env.action_spec.rand())
    stepped = torchrl_env.step(td)
    print("sample reward", stepped["next", "reward"], "done", stepped["next", "done"])

    move_fire_td = torchrl_env.reset()
    move_fire_td["move"] = torch.tensor(6, dtype=torch.int64)
    move_fire_td["aim"] = torch.tensor(0, dtype=torch.int64)
    move_fire_td["fire"] = torch.tensor(1, dtype=torch.int64)
    move_fire_next = torchrl_env.step(move_fire_td)
    print("decoded engine action", move_fire_next["next", "decoded_action"])

    check_env_specs = import_check_env_specs()
    if check_env_specs is not None:
        check_env_specs(torchrl_env)
        print("check_env_specs passed")
except Exception as exc:
    print("TorchRL wrapper demo skipped:", exc)

# PR3: PPO Smoke Training

This section proves the adapter can feed a minimal PPO loop. It is smoke validation, not policy-performance work.

In [ ]:
try:
    import torch
    from ppo_policy import MultiDiscreteActorCritic
    from train_ppo import PPOConfig, collect_rollout, train_ppo
    from eval_ppo import eval_checkpoint
    from torchrl_env import TorchRLCPCEnv

    env = TorchRLCPCEnv(seed=5, max_steps=8)
    policy = MultiDiscreteActorCritic(hidden_dim=32)
    obs = env.reset()
    output = policy.sample_action(obs)
    print("sampled policy action", {k: int(v) for k, v in output.action.items()})
    print("log_prob", float(output.log_prob.reshape(-1)[0]), "value", float(output.value.reshape(-1)[0]))

    rollout = collect_rollout(env, policy, PPOConfig(rollout_steps=8, max_episode_steps=8, hidden_dim=32))
    print("rollout obs shape", rollout["observations"].shape)

    cfg = PPOConfig(total_steps=32, rollout_steps=16, num_epochs=1, minibatch_size=8, max_episode_steps=8, hidden_dim=32)
    result = train_ppo(cfg)
    print("train result")
    print(json.dumps(result, indent=2))

    report = eval_checkpoint(result["checkpoint"], episodes=2)
    print("eval report")
    print(json.dumps(report, indent=2))
except Exception as exc:
    print("PPO smoke demo skipped:", exc)

# TODO PR4: scenario GT evaluation, trajectory export, and richer CPC metric reports.